# Validate the 1st order consensus against fake test data

This notebook validates the models fine-tuned in `finetune_original_models.ipynb` against fake test images.

Workflow:
1. Auto-discover the latest fine-tuned checkpoints from model registry
2. Run extraction jobs with those checkpoints
3. Build 1st-order consensus from extracted outputs
4. Score each extraction directly against ground truth

In [8]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Define model settings for validation jobs.
# Each entry: (label, model_slug, batch_size, total_shards)
MODEL_SETTINGS = [
    ("SmolVLM", "smolvlm2", 50, 1),
    ("Granite4", "granite4", 50, 1),
    ("Gemma3", "gemma3", 50, 1),
    ("Gemma4", "gemma4", 50, 1),
    ("Ministral", "ministral", 15, 1),
]

# ND96amsr A100 v4 has 8 GPUs per node. Use one extraction worker per GPU.
NODE_GPU_WORKERS = 8

# Must match the dataset used in finetune_original_models.ipynb
DATASET_NAME = "fake"
DATASET_DIR = "fake_daily_rainfall_2"

if DATASET_DIR:
    TRAINING_IMAGES_PATH = f"{DATASET_DIR.rstrip('/')}/images"
else:
    DATASET_IMAGES_MAP = {
        "real": "Daily_rainfall_sample/images",
        "fake": "fake_daily_rainfall/images",
        "test_real": "test_data/real/images",
        "test_fake": "test_data/fake/images",
    }
    if DATASET_NAME not in DATASET_IMAGES_MAP:
        raise ValueError(f"Unknown DATASET_NAME: {DATASET_NAME}")
    TRAINING_IMAGES_PATH = DATASET_IMAGES_MAP[DATASET_NAME]

print(f"Checkpoint training dataset path: {TRAINING_IMAGES_PATH}")
print(f"Node GPU workers per extraction job: {NODE_GPU_WORKERS}")

Checkpoint training dataset path: fake_daily_rainfall_2/images
Node GPU workers per extraction job: 8


## 1. Auto-discover fine-tuned checkpoints

This cell looks up the latest checkpoint for each model trained on the selected training dataset path.

In [9]:
def _first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


# Canonical registry location (shared across all notebooks and scripts)
REGISTRY_PATH = Path("../outputs/model_registry.json")
if not REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Registry not found: {REGISTRY_PATH.resolve()}")


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")


def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def _fix_checkpoint_path(path: str) -> str:
    """Keep canonical checkpoint path from the registry.
    
    aml_submit.sh resolves this path relative to AML_DATASTORE_BASE.
    Stripping Daily_rainfall_sample/ can break checkpoint mounts when
    the datastore base is rooted above that folder.
    """
    return _norm_rel(path)


registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
# Support both registry schemas (though now we only have one):
# 1) {"checkpoints": [...]} with keys model_slug/training_dataset_path
# 2) {"models": [...]} with keys base_model/dataset
entries = registry.get("checkpoints")
if not isinstance(entries, list):
    entries = registry.get("models", [])

MODEL_JOBS = []
missing = []
expected_dataset_norm = _norm_rel(TRAINING_IMAGES_PATH)

for label, model_slug, batch_size, total_shards in MODEL_SETTINGS:
    candidates = []
    for e in entries:
        entry_model = str(e.get("model_slug", e.get("base_model", "")))
        entry_dataset = str(
            e.get("training_dataset_path", e.get("dataset", ""))
        )
        checkpoint_path = e.get("checkpoint_path")

        if (
            entry_model == model_slug
            and _norm_rel(entry_dataset) == expected_dataset_norm
            and checkpoint_path
        ):
            candidates.append(e)

    if not candidates:
        missing.append(label)
        continue

    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )

    raw_checkpoint_path = str(best["checkpoint_path"])
    checkpoint_path = _fix_checkpoint_path(raw_checkpoint_path)
    MODEL_JOBS.append((label, checkpoint_path, batch_size, total_shards))
    print(f"{label}: {checkpoint_path}")

if missing:
    raise RuntimeError(
        "Missing fine-tuned checkpoints for: "
        + ", ".join(missing)
        + ". Check DATASET_DIR / DATASET_NAME and ensure model_registry has entries for this dataset."
    )

print(f"\nUsing canonical registry: {REGISTRY_PATH.resolve()}")
print(f"Discovered {len(MODEL_JOBS)} checkpoints")

SmolVLM: Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260622-125300/HuggingFaceTB--SmolVLM2-2.2B-Instruct
Granite4: Daily_rainfall_sample/outputs/checkpoints/granite4-20260622-125337/ibm-granite--granite-vision-4.1-4b
Gemma3: Daily_rainfall_sample/outputs/checkpoints/gemma3-20260622-125356/google--gemma-3-4b-it
Gemma4: Daily_rainfall_sample/outputs/checkpoints/gemma4-20260622-125416/google--gemma-4-E4B-it
Ministral: Daily_rainfall_sample/outputs/checkpoints/ministral-20260622-125437/mistralai--Mistral-Small-3.1-24B-Instruct-2503

Using canonical registry: /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/model_registry.json
Discovered 5 checkpoints


## 2. Run extraction jobs with fine-tuned checkpoints

This submits one extraction job per fine-tuned model checkpoint on fake test images.

In [10]:
for model_name, checkpoint, batch_size, total_shards in MODEL_JOBS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            "test_data/fake/images",
            "--transcriptions-path",
            "documents/DailyRainfall_UK/test_data/fake/transcriptions_0",
            "--total-shards",
            str(total_shards),
            "--node-gpu-workers",
            str(NODE_GPU_WORKERS),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

Submitting SmolVLM...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260622-125300/HuggingFaceTB--SmolVLM2-2.2B-Instruct
  Output:   azureml://subscriptions/79c7890c-2a30-44ef-a

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

jolly_floor_jp6sdftct3
Added extraction registry: ../outputs/extraction_registry.json
  Run: 20260623-134544
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../outputs/extraction_registry.json
Submitting Granite4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llm

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

helpful_eye_j56gmwzdly
Added extraction registry: ../outputs/extraction_registry.json
  Run: 20260623-134623
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../outputs/extraction_registry.json
Submitting Gemma3...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmda

...

plucky_dolphin_t77s5551l9
Added extraction registry: ../outputs/extraction_registry.json
  Run: 20260623-134642
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../outputs/extraction_registry.json
Submitting Gemma4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-ll

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

busy_match_chw9v6xpdb
Added extraction registry: ../outputs/extraction_registry.json
  Run: 20260623-134700
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../outputs/extraction_registry.json
Submitting Ministral...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/fake/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llm

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

joyful_van_f98yj2nb7v
Added extraction registry: ../outputs/extraction_registry.json
  Run: 20260623-134719
  Model: smolvlm
  Dataset: test_data/fake/images
Submitted 1 extract job(s).
Extraction registry: ../outputs/extraction_registry.json


When the jobs have been submitted, the extraction registry will contain run names needed for download and analysis.

Run the next cell to discover run names from the registry, then paste the printed `RUN_NAMES = [...]` block into the following cell.

In [ ]:
# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into the next cell.

EXTRACTION_REGISTRY_PATH = Path("../outputs/extraction_registry.json")
TARGET_IMAGES_PATH = "test_data/fake/images"

registry = json.loads(EXTRACTION_REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])

RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(TARGET_IMAGES_PATH)

for model_name, checkpoint, _batch_size, _total_shards in MODEL_JOBS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"])
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run extraction submission first, then re-run this cell."
    )

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]

print("\nCopy this block into the next cell:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")
print('EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]')

In [ ]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by the previous cell here.
RUN_NAMES = []

EXTRACTION_DIRS = [f"../outputs/extractions/{r}" for r in RUN_NAMES]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run the previous cell, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

When the jobs have completed successfully, download the extractions so we can analyze them locally.

In [ ]:
for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../scripts/aml_download.sh", "--run-name", run_name],
        check=True,
    )

Build 1st-order consensus from downloaded extractions and compare to ground truth.

This produces outputs in `outputs/consensus_fake_data/consensus_1st_order`.

In [ ]:
cmd = [
    "bash",
    "../scripts/run_consensus_pipeline.sh",
    "--dataset-root",
    "../outputs/consensus_fake_data",
    "--variant-name",
    "consensus_1st_order",
    "--threshold",
    "3",
    "--null-threshold",
    "5",
    "--precision",
    "3",
]

for extraction_dir in EXTRACTION_DIRS:
    cmd.extend(["--extraction-dir", extraction_dir])

cmd.extend(
    [
        "--validate",
        "--ground-truth-dir",
        "../test_data/fake/transcriptions",
        "--validation-sample-denominator",
        "1",
    ]
)

subprocess.run(cmd, check=True)

## 3. Score each individual extraction against ground truth

This computes per-model extraction quality directly against fake ground-truth transcriptions.

In [ ]:
labels = [name for name, _checkpoint, _batch, _shards in MODEL_JOBS]

cmd = [
    "python",
    "../scripts/evaluate_extraction_quality.py",
    "--ground-truth-dir",
    "../test_data/fake/transcriptions",
    "--output-json",
    "../outputs/consensus_fake_data/consensus_1st_order/extraction_quality_summary.json",
]

for label, extraction_dir in zip(labels, EXTRACTION_DIRS):
    cmd.extend(["--extraction-dir", extraction_dir, "--label", label])

subprocess.run(cmd, check=True)

print("\nSaved quality summary to:")
print(Path("../outputs/consensus_fake_data/consensus_1st_order/extraction_quality_summary.json").resolve())

In [ ]:
import json
from pathlib import Path


def _load_summary(path_str: str) -> list[dict]:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"Summary file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        if "results" in data and isinstance(data["results"], list):
            return data["results"]
        if "rows" in data and isinstance(data["rows"], list):
            return data["rows"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unexpected summary format in {path}")


def _index_by_label(rows: list[dict]) -> dict[str, dict]:
    out = {}
    for r in rows:
        label = str(r.get("label", "")).strip()
        if label:
            out[label] = r
    return out


summary_0th = _load_summary("../outputs/consensus_fake_data/consensus_0th_order/extraction_quality_summary.json")
summary_1st = _load_summary("../outputs/consensus_fake_data/consensus_1st_order/extraction_quality_summary.json")

by_0 = _index_by_label(summary_0th)
by_1 = _index_by_label(summary_1st)
labels = sorted(set(by_0.keys()) | set(by_1.keys()))

metrics = [
    "accuracy_vs_all_ground_truth_cells",
    "accuracy_on_evaluated_cells",
    "coverage_of_ground_truth_cells",
]

print("0TH vs 1ST ORDER EXTRACTION QUALITY")
print("=" * 120)
header = (
    f"{'Model':<12} "
    f"{'Acc(all) 0th':>12} {'Acc(all) 1st':>12} {'Delta':>10} "
    f"{'Acc(eval) 0th':>14} {'Acc(eval) 1st':>14} {'Delta':>10} "
    f"{'Coverage 0th':>12} {'Coverage 1st':>12} {'Delta':>10}"
)
print(header)
print("-" * len(header))

for label in labels:
    r0 = by_0.get(label, {})
    r1 = by_1.get(label, {})

    def _val(row: dict, key: str):
        v = row.get(key)
        return float(v) if isinstance(v, (int, float)) else None

    a0 = _val(r0, metrics[0])
    a1 = _val(r1, metrics[0])
    e0 = _val(r0, metrics[1])
    e1 = _val(r1, metrics[1])
    c0 = _val(r0, metrics[2])
    c1 = _val(r1, metrics[2])

    def _fmt(v):
        return f"{v:0.4f}" if isinstance(v, float) else "N/A"

    def _d(v1, v0):
        if isinstance(v1, float) and isinstance(v0, float):
            return v1 - v0
        return None

    print(
        f"{label:<12} "
        f"{_fmt(a0):>12} {_fmt(a1):>12} {_fmt(_d(a1, a0)):>10} "
        f"{_fmt(e0):>14} {_fmt(e1):>14} {_fmt(_d(e1, e0)):>10} "
        f"{_fmt(c0):>12} {_fmt(c1):>12} {_fmt(_d(c1, c0)):>10}"
    )

print("\nLegend: Delta = 1st_order - 0th_order")
print("Positive deltas indicate improvement in 1st-order over 0th-order.")